In [ ]:
import torch
from FNO1d import FNO1d  # 你自己的模型定义
import utilities3
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error

def list_pickle_files(folder_path):
    folder = Path(folder_path)
    # Recursively search for all .pkl files
    pickle_files = list(folder.rglob('*.pkl'))
    # Return absolute paths as strings
    return [str(f.resolve()) for f in pickle_files]

def load_checkpoint(path):
    torch.serialization.add_safe_globals([utilities3.UnitGaussianNormalizer])
    
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        # 完整checkpoint
        model = FNO1d(modes=checkpoint['modes'], width=checkpoint['width'])
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        return model, checkpoint['x_normalizer'], checkpoint['y_normalizer']
    else:
        # 纯参数字典
        model = FNO1d(modes=16, width=64)
        model.load_state_dict(checkpoint)
        model.eval()
        return model
    
def infer_two_models(input_tensor, model_norm, x_norm, y_norm, model_unnorm):
    # 归一化输入（normalized模型用）
    x_norm_encoded = x_norm.encode(input_tensor)
    x_norm_encoded = x_norm_encoded.to(torch.float32)  # 根据模型dtype调整

    # 直接用原始输入（unnormalized模型用）
    x_unnorm = input_tensor.to(torch.float32)

    with torch.no_grad():
        # normalized模型推断
        out_norm = model_norm(x_norm_encoded)
        out_norm = y_norm.decode(out_norm)

        # unnormalized模型推断
        out_unnorm = model_unnorm(x_unnorm)

    return out_norm, out_unnorm

def plot_inputs_and_outputs(x, pred1, pred2, y_true):
    if hasattr(x, 'cpu'):
        x = x.cpu().numpy()
    if hasattr(pred1, 'cpu'):
        pred1 = pred1.cpu().numpy()
    if hasattr(pred2, 'cpu'):
        pred2 = pred2.cpu().numpy()
    if hasattr(y_true, 'cpu'):
        y_true = y_true.cpu().numpy()

    # 计算 RMSE
    rmse1 = np.sqrt(mean_squared_error(y_true, pred1))
    rmse2 = np.sqrt(mean_squared_error(y_true, pred2))

    fig, axs = plt.subplots(2, 1, figsize=(10, 7))

    # 第一个图：输入 x
    axs[0].plot(x, label='Input')
    axs[0].set_title('Input')
    axs[0].legend()

    # 第二个图：两个模型输出 + 真实输出
    axs[1].plot(y_true, label='Ground Truth', linestyle='--', color='black')
    axs[1].plot(pred1, label=f'Model 1 Output (RMSE={rmse1:.4f})', alpha=0.8)
    axs[1].plot(pred2, label=f'Model 2 Output (RMSE={rmse2:.4f})', alpha=0.8)
    axs[1].set_title('Model Outputs vs Ground Truth')
    axs[1].legend()

    plt.tight_layout()
    plt.show()

normalized_path = "normalized/burgers_1d_FNO_model_trainedby_dim1d_nx1024_N1500_solver=exponax_kernel=gaussian_correlation_length0.03_bcperiodic_nu0.0005_t1.0_seed45.pth"
unnormalized_path = "unnormalized/burgers_1d_FNO_model_trainedby_dim1d_nx1024_N1500_solver=exponax_kernel=gaussian_correlation_length0.03_bcperiodic_nu0.0005_t1.0_seed45.pth"
model_norm, x_norm, y_norm = load_checkpoint(normalized_path)
model_unnorm = load_checkpoint(unnormalized_path)


: 

In [ ]:
input_tensor = torch.randn(20, 256, 1)  # 你自己的输入

out_norm, out_unnorm = infer_two_models(input_tensor, model_norm, x_norm, y_norm, model_unnorm)

print("Normalized model output shape:", out_norm.shape)
print("Unnormalized model output shape:", out_unnorm.shape)

In [ ]:

# Example usage:
folder = 'datasets'
all_pkl_files = list_pickle_files(folder)
for f in all_pkl_files:
    print(f)

: 